In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
mariamhany44_500_preprocessed_with_mask_path = kagglehub.dataset_download('mariamhany44/500-preprocessed-with-mask')

print('Data source import complete.')


100%|██████████| 791M/791M [00:38<00:00, 21.3MB/s]

Extracting files...


Data source import complete.


In [6]:
# =========================
# IMPORTS
# =========================
import os
import numpy as np
import pandas as pd
from pathlib import Path
import random
import json
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import LabelEncoder

# =========================
# CONFIG
# =========================
DATA_DIR = Path(mariamhany44_500_preprocessed_with_mask_path) / "splits_with_mask"
OUTPUT_DIR = Path("/content/results")
OUTPUT_DIR.mkdir(exist_ok=True)

BATCH_SIZE = 64
EPOCHS = 350
LR = 2e-4
PATIENCE = 40
MIN_DELTA = 0.001
SEED = 42
MIXUP_ALPHA = 0.3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

FEATURE_DIM = 438
D_MODEL = 160
N_HEADS = 4
N_LAYERS = 3

MIN_SAMPLES_TARGET = 50

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(DATA_DIR / "splits.csv")

def fix_path(p):
    return DATA_DIR / p.replace("\\", "/")

df["filepath"] = df["filepath"].apply(fix_path)
df["maskpath"] = df["maskpath"].apply(fix_path)

le = LabelEncoder()
df["label_id"] = le.fit_transform(df["label"])
num_classes = len(le.classes_)

np.save(OUTPUT_DIR / "label_classes.npy", le.classes_)

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

# =========================
# AUGMENTATION (SMART)
# =========================
class Augmenter:
    def __init__(self):
        self.LH_START = 312
        self.RH_START = 375

    def __call__(self,x):
        if random.random()<0.5:
            idx = np.linspace(0,len(x)-1,125,dtype=int)
            x = x[idx]

        if random.random()<0.7:
            x += np.random.normal(0,0.008,x.shape)

        if random.random()<0.6:
            dx,dy = random.uniform(-0.02,0.02),random.uniform(-0.02,0.02)
            x[:,312:375:3]+=dx
            x[:,375:438:3]+=dx

        return x.astype(np.float32)

augmenter = Augmenter()

# =========================
# DATASET
# =========================
class ASLDataset(Dataset):
    def __init__(self, df, train=True):
        self.df = df
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = np.load(row["filepath"])
        mask = np.load(row["maskpath"])
        x = x * mask[:, None]

        if self.train:
            x = augmenter(x)

        y = row["label_id"]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y), str(row["filepath"])

# =========================
# SAMPLER (BALANCED)
# =========================
class_counts = Counter(train_df["label_id"])

weights = []
for lbl in train_df["label_id"]:
    count = class_counts[lbl]
    eff = max(count, MIN_SAMPLES_TARGET)
    weights.append(1.0 / eff)

sampler = WeightedRandomSampler(weights, len(weights))

train_loader = DataLoader(ASLDataset(train_df, True), batch_size=BATCH_SIZE, sampler=sampler)
val_loader   = DataLoader(ASLDataset(val_df, False), batch_size=BATCH_SIZE)
test_loader  = DataLoader(ASLDataset(test_df, False), batch_size=BATCH_SIZE)

# =========================
# MODEL (SOTA STYLE)
# =========================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=300):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2)*(-np.log(10000.0)/d_model))
        pe[:,0::2]=torch.sin(pos*div)
        pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self,x):
        return x + self.pe[:, :x.size(1)]

class MotionExtractor(nn.Module):
    def forward(self,x):
        dx = x[:,1:] - x[:,:-1]
        pad = torch.zeros_like(dx[:,0:1])
        return torch.cat([pad, dx], dim=1)

class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.motion = MotionExtractor()

        self.pose = nn.Linear(132*2, 32)
        self.face = nn.Linear(180*2, 32)
        self.lh   = nn.Linear(63*2, 32)
        self.rh   = nn.Linear(63*2, 32)

        self.hand_attention = nn.MultiheadAttention(64, 4, batch_first=True)

        self.fusion = nn.Linear(32*4, D_MODEL)

        self.cls = nn.Parameter(torch.zeros(1,1,D_MODEL))
        self.pos = PositionalEncoding(D_MODEL)

        enc = nn.TransformerEncoderLayer(D_MODEL, N_HEADS, D_MODEL*4, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, N_LAYERS)

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.fc = nn.Sequential(
            nn.LayerNorm(D_MODEL),
            nn.Linear(D_MODEL, num_classes)
        )

    def forward(self,x):
        motion = self.motion(x)
        x = torch.cat([x, motion], dim=-1)

        pose = self.pose(x[:,:,:132*2])
        face = self.face(x[:,:,132*2:312*2])
        lh   = self.lh(x[:,:,312*2:375*2])
        rh   = self.rh(x[:,:,375*2:438*2])

        hands = torch.cat([lh,rh], dim=-1)
        hands,_ = self.hand_attention(hands, hands, hands)
        lh, rh = hands.chunk(2, dim=-1)

        x = torch.cat([pose,face,lh,rh], dim=-1)
        x = self.fusion(x)

        B = x.size(0)
        cls = self.cls.expand(B,-1,-1)
        x = torch.cat([cls,x],dim=1)

        x = self.pos(x)
        x = self.encoder(x)

        pooled = self.pool(x.transpose(1,2)).squeeze(-1)

        return self.fc(pooled)

model = Model().to(DEVICE)

# =========================
# LOSS
# =========================
counts = np.array([class_counts[i] for i in range(num_classes)])
weights_tensor = torch.tensor(1.0/counts, dtype=torch.float32).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=weights_tensor, label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)

# =========================
# TRAIN LOOP
# =========================
best_val_acc = 0
patience_counter = 0

for epoch in range(EPOCHS):

    model.train()
    train_correct=0
    total=0

    for x,y,_ in train_loader:
        x,y=x.to(DEVICE),y.to(DEVICE)

        lam = np.random.beta(MIXUP_ALPHA,MIXUP_ALPHA)
        perm = torch.randperm(x.size(0)).to(DEVICE)

        x_mix = lam*x + (1-lam)*x[perm]
        y_a,y_b=y,y[perm]

        optimizer.zero_grad()
        outputs=model(x_mix)

        loss = lam*criterion(outputs,y_a)+(1-lam)*criterion(outputs,y_b)
        loss.backward()
        optimizer.step()

        preds = outputs.argmax(1)
        train_correct += (
            lam*(preds==y_a).sum().item()+
            (1-lam)*(preds==y_b).sum().item()
        )
        total+=y.size(0)

    train_acc=train_correct/total

    model.eval()
    val_correct=0
    total_val=0

    with torch.no_grad():
        for x,y,_ in val_loader:
            x,y=x.to(DEVICE),y.to(DEVICE)
            preds=model(x).argmax(1)
            val_correct+=(preds==y).sum().item()
            total_val+=y.size(0)

    val_acc=val_correct/total_val

    print(f"\nEpoch {epoch+1}")
    print(f"Train Acc {train_acc:.4f} | Val Acc {val_acc:.4f}")

    if val_acc > best_val_acc + MIN_DELTA:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), OUTPUT_DIR / "best_model.pth")
    else:
        patience_counter+=1
        if patience_counter>=PATIENCE:
            print("Early stopping")
            break

# =========================
# TEST
# =========================
model.load_state_dict(torch.load(OUTPUT_DIR / "best_model.pth"))
model.eval()

correct=0
total=0
wrong=[]

with torch.no_grad():
    for x,y,paths in test_loader:
        x,y=x.to(DEVICE),y.to(DEVICE)
        preds=model(x).argmax(1)

        correct+=(preds==y).sum().item()
        total+=y.size(0)

        for i in range(len(y)):
            if preds[i]!=y[i]:
                wrong.append({
                    "file": paths[i],
                    "predicted": le.classes_[preds[i].item()],
                    "truth": le.classes_[y[i].item()]
                })

test_acc = correct/total
print("\nFINAL TEST ACC:", test_acc)

pd.DataFrame(wrong).to_csv(OUTPUT_DIR/"wrong_predictions.csv", index=False)

metrics = {
    "test_accuracy": test_acc,
    "best_val_accuracy": best_val_acc
}
with open(OUTPUT_DIR/"metrics.json","w") as f:
    json.dump(metrics,f,indent=4)

print("Saved everything to /kaggle/working/")


Epoch 1
Train Acc 0.0022 | Val Acc 0.0019

Epoch 2
Train Acc 0.0027 | Val Acc 0.0047

Epoch 3
Train Acc 0.0112 | Val Acc 0.0159

Epoch 4
Train Acc 0.0180 | Val Acc 0.0248

Epoch 5
Train Acc 0.0402 | Val Acc 0.0552

Epoch 6
Train Acc 0.0614 | Val Acc 0.0833

Epoch 7
Train Acc 0.0881 | Val Acc 0.1198

Epoch 8
Train Acc 0.1299 | Val Acc 0.1488

Epoch 9
Train Acc 0.1572 | Val Acc 0.1432

Epoch 10
Train Acc 0.1948 | Val Acc 0.2068

Epoch 11
Train Acc 0.2266 | Val Acc 0.2284

Epoch 12
Train Acc 0.2819 | Val Acc 0.2934

Epoch 13
Train Acc 0.2931 | Val Acc 0.3542

Epoch 14
Train Acc 0.3581 | Val Acc 0.3730

Epoch 15
Train Acc 0.4065 | Val Acc 0.4141

Epoch 16
Train Acc 0.4163 | Val Acc 0.4310

Epoch 17
Train Acc 0.4693 | Val Acc 0.4600

Epoch 18
Train Acc 0.5120 | Val Acc 0.5124

Epoch 19
Train Acc 0.5162 | Val Acc 0.5302

Epoch 20
Train Acc 0.5326 | Val Acc 0.5526

Epoch 21
Train Acc 0.5372 | Val Acc 0.5625

Epoch 22
Train Acc 0.6006 | Val Acc 0.5788

Epoch 23
Train Acc 0.5990 | Val Acc 0.57